In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
data = pd.read_csv(
    "D:\\Sem-6 Labs\\ML\\Project\\backend\\data\\clean_cardio_data.csv",
    sep=","
)


data.head()

,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,gender,active,age,cardio,bmi
0,168,62.0,110,80,1,1,0,0,2,1,50,0,21.967120
1,156,85.0,140,90,3,1,0,0,1,1,55,1,34.927679
2,165,64.0,130,70,3,1,0,0,1,0,51,1,23.507805
3,169,82.0,150,100,1,1,0,0,2,1,48,1,28.710479
4,156,56.0,100,60,1,1,0,0,1,0,47,0,23.011177


## Model

### Split data into Train and Test

In [3]:
from sklearn.model_selection import train_test_split

x=data.drop(["cardio"],axis=1)
y=data["cardio"]

X_train,X_test,Y_train,Y_test = train_test_split(x,y,test_size=0.2,random_state=42)

### Shape of Dataset, Training & Testing Dataset

In [4]:
print(f"Dataset Shape : {data.shape}")
print(f"Training Shape : {X_train.shape}")
print(f"Testing Shape : {X_test.shape}")

Dataset Shape : (68576, 13)
Training Shape : (54860, 12)
Testing Shape : (13716, 12)


### Transform Data

In [5]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)



### Apply Random Forest and Train Model 

In [ ]:
model = RandomForestClassifier(random_state=42)

In [12]:
from sklearn.model_selection import StratifiedKFold


param_grid = {
    'n_estimators': [200],
    'max_depth': [None, 20],
    'min_samples_split': [2],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    model,
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, Y_train)

print("Best Parameters:")
print(grid_search.best_params_)

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Parameters:
{'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}


In [21]:
best_model = grid_search.best_estimator_

### Predict Values

In [16]:
Y_pred = best_model.predict(X_test)
Y_train_pred = best_model.predict(X_train)

### Accuracy of Model

In [17]:
print("Testing Accuracy:", round(accuracy_score(Y_test, Y_pred)*100, 2), "%")


Testing Accuracy: 73.1 %


In [ ]:
print("Training Accuracy:", round(accuracy_score(Y_train, Y_train_pred)*100, 2), "%") 


Training Accuracy: 85.03 %


### Classfication

In [19]:
print("Classification Report:")
print(classification_report(Y_test, Y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.76      0.74      6954
           1       0.74      0.70      0.72      6762

    accuracy                           0.73     13716
   macro avg       0.73      0.73      0.73     13716
weighted avg       0.73      0.73      0.73     13716



### Confusion Matrix

In [20]:
from sklearn.metrics import confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(Y_test, Y_pred))

Confusion Matrix:
[[5283 1671]
 [2019 4743]]
